# Algebras and elementary functions with NumPy

`algpy` supplies scalar number types that also work as NumPy dtypes.
This tour starts with familiar quaternion arithmetic and reaches all three
general families with **five generators and 32 real coefficients**.

Install the examples environment with `python -m pip install -e ".[examples]"`
from the repository root, then run this notebook from top to bottom.
The last figure has an interactive slider.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.offline import init_notebook_mode
from plotly.subplots import make_subplots
from IPython.display import Markdown, display

from algpy.algebras import quaternion, complex1, splitcomplex1, dual1, dual5
from algpy.algebras.general import cayleydickson, clifford, multicomplex
from _plotting import slider_controls

init_notebook_mode(connected=False)
pio.templates.default = "plotly_white"

## One scalar type, the usual arithmetic

Omitted constructor coefficients are zero. `gens()` excludes the identity;
`basis()` includes it. Components are indexed by bit masks of generators.
Quaternion multiplication is noncommutative.

In [ ]:
i, j = quaternion.gens()
k = i*j
assert i*i == j*j == k*k == quaternion(-1)
assert i*j == -(j*i)

q = quaternion(1.3, .15, .2, -.1)
np.testing.assert_allclose(quaternion.to_float(q*q.inv()), [1, 0, 0, 0], atol=1e-14)
print("q =", q)
print("generators:", quaternion.gens())
print("basis coefficients:\n", quaternion.to_float(quaternion.basis()))
print("norm:", q.norm(), type(q.norm()).__name__)

## Arrays, broadcasting, and coefficient conversion

`from_float` consumes a final coefficient axis. `to_float` restores it.
The algebra's dtype is retained by NumPy operations and broadcasting.

In [ ]:
rows = np.array([[1.3, .15, .2, -.1], [.8, -.1, .05, .2]])
values = quaternion.from_float(rows)
exponentials = np.exp(values)
broadcast = values[:, None] * np.array([.5, 1., 2.])
assert values.dtype == quaternion.dtype
assert broadcast.shape == (2, 3)
np.testing.assert_array_equal(quaternion.to_float(values), rows)
print("algebra shape:", values.shape, "coefficient shape:", quaternion.to_float(values).shape)
print("exp coefficients:\n", quaternion.to_float(exponentials))

## Elementary functions

NumPy ufuncs and scalar methods evaluate the same algebra functions.
Inverse functions use principal branches. A round trip such as
`log(exp(q))` needs the input to stay within that branch.

In [ ]:
operations = ["exp", "expm1", "log", "log1p", "sqrt", "sin", "cos", "tan",
              "arcsin", "arccos", "arctan", "sinh", "cosh", "tanh",
              "arcsinh", "arccosh", "arctanh"]
table = ["| NumPy function | Quaternion coefficients |", "| --- | --- |"]
for name in operations:
    result = getattr(np, name)(q)
    assert np.isfinite(quaternion.to_float(result)).all()
    table.append(f"| `np.{name}` | `{np.round(quaternion.to_float(result), 5)}` |")
display(Markdown("\n".join(table)))
np.testing.assert_allclose(quaternion.to_float(q.exp()), quaternion.to_float(np.exp(q)))
np.testing.assert_allclose(quaternion.to_float(np.log(np.exp(q))), rows[0], atol=1e-14)
np.testing.assert_allclose(quaternion.to_float(np.square(np.sqrt(q))), rows[0], atol=1e-14)

## Left powers, right powers, and integer powers

With a noncommutative algebra, the order matters:
\[
a.\mathrm{lpow}(b)=\exp(b\log a),\qquad
a.\mathrm{rpow}(b)=\exp((\log a)b).
\]
`a**b` and `np.power(a, b)` use right power. `ipow` takes an integer
exponent directly; it is also useful when a logarithm is undefined.

In [ ]:
base = quaternion(1.2, .4)
exponent = quaternion(.4, 0, .3)
left, right = base.lpow(exponent), base.rpow(exponent)
assert not np.allclose(quaternion.to_float(left), quaternion.to_float(right))
np.testing.assert_allclose(quaternion.to_float(base**exponent), quaternion.to_float(right))
np.testing.assert_allclose(quaternion.to_float(base.ipow(3)), quaternion.to_float((base*base)*base))
assert quaternion.zero().ipow(0) == quaternion.one()
print("left power:", left)
print("right power:", right)

## Five-generator general algebras

Each constructor takes generator squares. These examples use five `-1`
parameters. Clifford multiplication is associative; multicomplex
multiplication is also commutative. Cayley–Dickson multiplication at this
dimension is nonassociative, so parentheses matter.

`minpoly()` returns the **generic minimal polynomial**, with ascending
coefficients and retained multiplicities. It does not reduce the degree
for a particular scalar or repeated-root element. For nonassociative
algebras, its roots need not be the full spectrum of `lmat()`.

In [ ]:
families = {
    "Cayley–Dickson": cayleydickson(-1, -1, -1, -1, -1),
    "Clifford": clifford(-1, -1, -1, -1, -1),
    "Multicomplex": multicomplex(-1, -1, -1, -1, -1),
}
table = ["| Family | Components | Generators | Generic polynomial degree | Norm type |",
         "| --- | ---: | ---: | ---: | --- |"]
for name, algebra in families.items():
    value = algebra(1.2, *np.linspace(-.03, .03, 31))
    assert algebra.N == 32 and len(algebra.gens()) == 5
    np.testing.assert_allclose(algebra.to_float(np.log(np.exp(value))), algebra.to_float(value),
                               rtol=2e-10, atol=2e-12)
    table.append(f"| {name} | {algebra.N} | {len(algebra.gens())} | "
                 f"{len(value.minpoly())-1} | `{type(value.norm()).__name__}` |")
display(Markdown("\n".join(table)))

cd = families["Cayley–Dickson"]
a, b, c = cd.gens()[:3]
assert (a*b)*c != a*(b*c)
mc = families["Multicomplex"]
a, b = mc.gens()[:2]
assert a*b == b*a

## Multiplication matrices and polynomial metadata

`lmat()` acts on coefficient **columns**; `rmat()` acts on **rows**.
These two conventions make both expressions below equal the coefficients
of the algebra product. An associative algebra also gives a matrix
representation of its products. That extra property does not hold for
general Cayley–Dickson algebras.

In [ ]:
a, b = q, quaternion(.8, -.1, .05, .2)
product = quaternion.to_float(a*b)
np.testing.assert_allclose(a.lmat() @ quaternion.to_float(b), product)
np.testing.assert_allclose(quaternion.to_float(a) @ b.rmat(), product)
polynomial = a.minpoly()
residual = np.square(a) + polynomial[1]*a + polynomial[0]
np.testing.assert_allclose(quaternion.to_float(residual), 0, atol=1e-14)
print("left multiplication matrix:\n", a.lmat())
print("ascending quadratic coefficients:", polynomial)
print("quadratic roots:", a.eivals())

## A fifth mixed derivative in one evaluation

The five dual generators commute and square to zero. Seeding the same
real input in all five directions puts its fifth derivative in coefficient
`0b11111 = 31`. No finite-difference step size is involved.

In [ ]:
epsilon = dual5.gens()
seeded = dual5(.3)
for generator in epsilon:
    seeded = seeded + generator
result = np.exp(seeded)
np.testing.assert_allclose(result[31], np.exp(.3), rtol=1e-13)
print("exp(.3):", result[0], "fifth mixed derivative:", result[31])

## The same exponential, three generator squares

For one generator $e$, its square determines the shape of $\exp(te)$:
a circle for $e^2=-1$, a hyperbola for $e^2=+1$, and a straight line for
$e^2=0$. The plotted coordinates are the scalar and generator coefficients.
Move the slider or hover over a curve.

In [ ]:
parameters = np.linspace(-1.2, 1.2, 49)
one_generator = [(complex1, "e² = −1", "#2563eb"),
                 (splitcomplex1, "e² = +1", "#db2777"),
                 (dual1, "e² = 0", "#059669")]
tracks = []
fig = make_subplots(rows=1, cols=3, subplot_titles=[label for _, label, _ in one_generator])
for column, (algebra, label, color) in enumerate(one_generator, 1):
    seeds = algebra.from_float(np.column_stack([np.zeros_like(parameters), parameters]))
    coordinates = algebra.to_float(np.exp(seeds))
    tracks.append(coordinates)
    fig.add_trace(go.Scatter(x=coordinates[:, 0], y=coordinates[:, 1],
                            mode="lines", line_color=color, name=label), row=1, col=column)
    fig.add_trace(go.Scatter(x=[coordinates[0, 0]], y=[coordinates[0, 1]],
                            mode="markers", marker=dict(size=11, color=color),
                            showlegend=False), row=1, col=column)
    fig.update_xaxes(title_text="scalar coefficient", range=[-.1, 2.1], row=1, col=column)
    fig.update_yaxes(title_text="generator coefficient", scaleanchor="x" if column == 1 else f"x{column}",
                     scaleratio=1, range=[-1.7, 1.7], row=1, col=column)
fig.frames = [go.Frame(name=str(index), traces=[1, 3, 5], data=[
    go.Scatter(x=[track[index, 0]], y=[track[index, 1]]) for track in tracks
]) for index in range(len(parameters))]
fig.update_layout(title="Exponential paths in three two-component algebras", height=450)
slider_controls(fig, [f"{t:.2f}" for t in parameters]).show(auto_play=False)

**Next:** open `02_projective_geometry.ipynb` for meets, joins, and motor
interpolation, or `04_curves_and_dual_numbers.ipynb` to unpack the derivative
coefficients geometrically.